In [ ]:
# ============================================
# E. coli Resistance (Kaggle) – ML Pipeline
# Single-antibiotic binary task with baselines + 5-fold CV
# ============================================

#benchmark: https://www.nature.com/articles/s41598-020-71693-5

!pip install -q kagglehub pandas numpy scikit-learn matplotlib seaborn

import os
import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# --------------------------------------------
# 1. Load Kaggle dataset
# --------------------------------------------

path = kagglehub.dataset_download("valeriamaciel/e-coli-resistance-dataset")
print("Path to dataset files:", path)

csv_path = os.path.join(path, "BVBRC_E.coli_Dataset.csv")
df = pd.read_csv(csv_path, low_memory=False)

print("\nRaw shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nHead:")
print(df.head())

# --------------------------------------------
# 2. Basic cleaning and label definition
# --------------------------------------------

df["Resistant Phenotype"] = df["Resistant Phenotype"].astype(str).str.strip()

valid_labels = ["Resistant", "Susceptible"]
df_clean = df[df["Resistant Phenotype"].isin(valid_labels)].copy()
print("\nAfter filtering to clear R/S phenotypes:", df_clean.shape)

print("\nTop antibiotics by count:")
print(df_clean["Antibiotic"].value_counts().head(20))

target_ab = "ciprofloxacin"  # choose an antibiotic with many R and S
df_ab = df_clean[df_clean["Antibiotic"].str.lower() == target_ab.lower()].copy()
print(f"\nSubset for antibiotic = {target_ab}: shape {df_ab.shape}")

df_ab["y"] = (df_ab["Resistant Phenotype"] == "Resistant").astype(int)
print("\nLabel counts (0=Susceptible, 1=Resistant):")
print(df_ab["y"].value_counts())

# --------------------------------------------
# 3. Feature selection and encoding
# --------------------------------------------

feature_cols = [
    "Genome ID",
    "Measurement Sign",
    "Measurement Value",
    "Measurement Unit",
    "Laboratory Typing Method",
    "Laboratory Typing Platform",
    "Vendor",
    "Testing Standard",
    "Testing Standard Year",
    "Source",
]

df_ab_feat = df_ab[feature_cols + ["y"]].copy()

numeric_cols = ["Measurement Value", "Testing Standard Year"]
for c in numeric_cols:
    df_ab_feat[c] = pd.to_numeric(df_ab_feat[c], errors="coerce")

for c in numeric_cols:
    df_ab_feat[c] = df_ab_feat[c].fillna(df_ab_feat[c].median())

cat_cols = [c for c in feature_cols if c not in numeric_cols]
for c in cat_cols:
    df_ab_feat[c] = df_ab_feat[c].astype(str).fillna("Unknown")

X_cat = df_ab_feat[cat_cols]
X_num = df_ab_feat[numeric_cols]
y = df_ab_feat["y"].values

# Updated OneHotEncoder for new sklearn
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat_ohe = ohe.fit_transform(X_cat)

X_num_mat = sparse.csr_matrix(X_num.values)
X = sparse.hstack([X_num_mat, X_cat_ohe]).tocsr()

print("\nFinal feature matrix shape:", X.shape)

# --------------------------------------------
# 4. Train/test split (held-out evaluation)
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nTrain label counts:", np.bincount(y_train))
print("Test label counts:", np.bincount(y_test))

# --------------------------------------------
# 5. Baseline models
# --------------------------------------------

# Majority-class baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"\n=== {target_ab}: Dummy (majority class) baseline ===")
print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("F1 (binary, pos=Resistant):", f1_score(y_test, y_pred_dummy, average="binary", pos_label=1))

# Simple Decision Tree baseline
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print(f"\n=== {target_ab}: Decision Tree baseline ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("F1 (binary, pos=Resistant):", f1_score(y_test, y_pred_tree, average="binary", pos_label=1))

# --------------------------------------------
# 6. 5-fold stratified CV – Logistic Regression
# --------------------------------------------

log_reg = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced"
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_lr = cross_validate(
    log_reg,
    X, y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print(f"\n=== {target_ab}: Logistic Regression (class-weighted) – 5-fold CV ===")
for metric in scoring.keys():
    scores = cv_results_lr[f"test_{metric}"]
    print(f"{metric}: mean={scores.mean():.3f}, std={scores.std():.3f}")

# Fit on full training, evaluate on held-out test
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

print(f"\n=== {target_ab}: Logistic Regression – held-out test ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_lr, zero_division=0))
print("F1:", f1_score(y_test, y_pred_lr, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print(classification_report(y_test, y_pred_lr))

# --------------------------------------------
# 7. 5-fold stratified CV – Random Forest
# --------------------------------------------

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

cv_results_rf = cross_validate(
    rf,
    X, y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print(f"\n=== {target_ab}: Random Forest (class-weighted) – 5-fold CV ===")
for metric in scoring.keys():
    scores = cv_results_rf[f"test_{metric}"]
    print(f"{metric}: mean={scores.mean():.3f}, std={scores.std():.3f}")

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(f"\n=== {target_ab}: Random Forest – held-out test ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1:", f1_score(y_test, y_pred_rf, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))

100%|██████████| 3.03M/3.03M [00:00<00:00, 60.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/valeriamaciel/e-coli-resistance-dataset/versions/4

Raw shape: (195795, 21)

Columns: ['Taxon ID', 'Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'Measurement', 'Measurement Sign', 'Measurement Value', 'Measurement Unit', 'Laboratory Typing Method', 'Laboratory Typing Method Version', 'Laboratory Typing Platform', 'Vendor', 'Testing Standard', 'Testing Standard Year', 'Computational Method', 'Computational Method Version', 'Computational Method Performance', 'Evidence', 'Source', 'PubMed']

Head:
   Taxon ID   Genome ID                                        Genome Name  \
0       562  562.144628                                Escherichia coli 10   
1       562  562.569100                      Escherichia coli CVM N36113PS   
2       562  562.144245                        Escherichia coli ERR7221502   
3       562  562.100003  Escherichia coli c1b4da3e-7bb9-11e9-a8d3-68b59...   
4       562  562.658140            